### GPU Check

This cell checks whether an NVIDIA GPU is available in the training environment.  
The `nvidia-smi` command displays information about the GPU such as the model, driver version, CUDA version, and current memory usage.

Verifying GPU availability is important because training transformer models like **GPT-2 Medium** requires significant computational power. Running the training on a GPU dramatically reduces the training time compared to using a CPU.

In [ ]:
!nvidia-smi

### Loading the Pretrained GPT-2 Medium Model

This cell loads the **GPT-2 Medium** model and its corresponding tokenizer from the Hugging Face Transformers library.

The **tokenizer** is responsible for converting text into numerical tokens that the model can understand.  
The **GPT2LMHeadModel** is the language model used for text generation and will later be fine-tuned on the custom instruction dataset.

By loading a pretrained model instead of training from scratch, we take advantage of the knowledge already learned from large-scale text corpora and adapt it to our specific task (MIT-WPU assistant).

After loading both the tokenizer and the model, a confirmation message is printed to indicate that the download and initialization were successful.

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_name = "gpt2-medium"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

print("Model downloaded successfully")

### Saving the Downloaded Model and Tokenizer

This cell saves the loaded **GPT-2 Medium model** and its corresponding **tokenizer** to a local directory.

Saving the model locally ensures that it does not need to be downloaded again from the Hugging Face servers in future sessions. This is especially useful when working in environments like Google Colab where sessions can reset.

Both the model weights and tokenizer configuration are stored in the specified directory, allowing the model to be easily reloaded later for fine-tuning, inference, or deployment.

In [ ]:
model.save_pretrained("/content/gpt2-medium")
tokenizer.save_pretrained("/content/gpt2-medium")

### Loading the Locally Saved Model and Tokenizer

This cell loads the **GPT-2 Medium model** and its corresponding **tokenizer** from the local directory where they were previously saved.

Instead of downloading the model again from Hugging Face, the `from_pretrained()` method reads the model weights and tokenizer configuration directly from the specified local path. This makes the workflow faster and ensures that the exact same model version is used in the next stages of the pipeline.

Loading the model from local storage is particularly useful when continuing training, running experiments, or performing inference after the model has already been downloaded or fine-tuned.

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model = GPT2LMHeadModel.from_pretrained("/content/gpt2-medium")
tokenizer = GPT2Tokenizer.from_pretrained("/content/gpt2-medium")

### Testing the Loaded Model with a Sample Prompt

This cell performs a quick test to verify that the loaded **GPT-2 Medium model** is working correctly for text generation.

A sample prompt is first defined and passed through the **tokenizer**, which converts the input text into token IDs that the model can process. The `model.generate()` function is then used to produce a continuation of the prompt.

Several generation parameters are used:

- **max_length** – limits the maximum number of tokens in the generated output  
- **num_return_sequences** – specifies how many generated responses to produce  
- **temperature** – controls randomness in the output (higher values produce more diverse text)  
- **do_sample** – enables probabilistic sampling instead of deterministic generation  

Finally, the generated tokens are **decoded back into readable text** and printed. This step helps confirm that the model and tokenizer are properly loaded and capable of generating coherent text.

In [ ]:
import torch

prompt = "Artificial intelligence will"

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_length=50,
    num_return_sequences=1,
    temperature=0.8,
    do_sample=True
)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(text)

### Loading and Combining the Training Datasets

This cell loads multiple JSON files that contain the instruction dataset used for fine-tuning the model. Each file represents a different category of training data such as identity responses, general conversations, RAG examples, tool calling, and tool fallback cases.

All dataset files are listed in the `files` array. The code then iterates through each file, opens it, and loads the JSON content using Python's `json` module. The samples from every file are appended into a single list called `data`.

Combining all files into one list creates a unified dataset that will later be used for preprocessing and model training.

Finally, the cell prints:
- The **total number of samples** loaded from all dataset files
- The **first sample in the dataset** to verify that the data structure is correct.

In [ ]:
import json

files = [
    "Identity.json",
    "general.json",
    "rag.json",
    "tool.json",
    "tool_fallback.json"
]

data = []

for file in files:
    with open(file, "r") as f:
        data.extend(json.load(f))

print("Total samples:", len(data))
print(data[0])

### Formatting Training Examples

This cell defines a helper function that converts each dataset entry into a **text format suitable for training the language model**.

The function takes a single dataset sample and transforms it into a structured conversation format using special role tags:

- `<|user|>` represents the user's instruction or question  
- `<|assistant|>` represents the model's expected response  

If the example type is **RAG**, the function also includes the retrieved **context** before the conversation. This teaches the model to generate answers based on provided information.

For all other example types (general conversation, identity, tool calling, and fallback), the function simply formats the interaction as a user message followed by the assistant response.

This formatting step ensures that all training samples follow a **consistent conversational structure**, which helps the model learn how to respond appropriately during fine-tuning.

In [ ]:
def format_example(example):

    if example["type"] == "rag":
        return f"""Context:
{example['context']}

<|user|>: {example['instruction']}
<|assistant|>: {example['output']}"""

    else:
        return f"<|user|>: {example['instruction']}\n<|assistant|>: {example['output']}"


### Preparing and Tokenizing the Dataset for Training

These cells convert the formatted training examples into a tokenized dataset that can be used to train the GPT-2 model.

First, all formatted examples are collected into a list called `texts`. A sample is printed to verify that the formatting function produced the expected conversation structure.

The tokenizer is then loaded from the previously saved GPT-2 Medium directory. Two additional special tokens are introduced:

- `<|user|>` to represent the user instruction
- `<|assistant|>` to represent the assistant response

These tokens help the model learn the conversational structure of the dataset. Since new tokens are added, the model's embedding layer is resized to match the updated tokenizer vocabulary.

Next, the dataset is converted into a Hugging Face `Dataset` object, which provides efficient utilities for preprocessing and training.

A custom `tokenize` function is defined to prepare each sample for training:

- Text is tokenized with truncation and padding to a fixed maximum length.
- The `<|assistant|>` token is located in each sequence.
- Tokens that belong to the **user prompt** are masked by setting their label values to `-100`.

Masking the user portion ensures that the model only learns to predict the **assistant's response**, which is the desired behavior during instruction fine-tuning.

Finally, the dataset is shuffled for better training randomness and the tokenization function is applied to the entire dataset using the `map()` method.

In [ ]:
texts = [format_example(x) for x in data]
print(texts[0])

In [ ]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("/content/gpt2-medium")

special_tokens = {
    "additional_special_tokens": ["<|user|>", "<|assistant|>"]
}

tokenizer.add_special_tokens(special_tokens)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel.from_pretrained("/content/gpt2-medium")

model.resize_token_embeddings(len(tokenizer))

In [ ]:
from datasets import Dataset

dataset = Dataset.from_dict({"text": texts})

In [ ]:
def tokenize(batch):

    tokens = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    assistant_token_id = tokenizer.convert_tokens_to_ids("<|assistant|>")

    labels = []

    for seq in tokens["input_ids"]:

        label_seq = seq.copy()

        if assistant_token_id in seq:
            idx = seq.index(assistant_token_id)

            for j in range(idx + 1):
                label_seq[j] = -100

        labels.append(label_seq)

    tokens["labels"] = labels

    return tokens

dataset = dataset.shuffle(seed=42)

dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

### Defining Training Configuration

This cell defines the training configuration used for fine-tuning the model using the Hugging Face `TrainingArguments` class. These parameters control how the model will be trained, logged, and saved during the training process.

Key parameters used:

- **output_dir** – Directory where training outputs such as checkpoints and logs will be stored.
- **per_device_train_batch_size** – Number of samples processed per GPU during each training step.
- **num_train_epochs** – Number of times the model will iterate over the entire training dataset.
- **learning_rate** – Controls how much the model weights are updated during each optimization step.
- **logging_steps** – Determines how frequently training progress (loss, metrics) is logged.
- **save_strategy** – Controls checkpoint saving behavior. Setting it to `"no"` disables intermediate checkpoint saving.
- **fp16** – Enables mixed precision training (16-bit floating point), which reduces GPU memory usage and can speed up training on supported hardware.

These settings define the overall behavior of the training process before the model is passed to the training loop.

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./energy_model",

    per_device_train_batch_size=4,
    num_train_epochs=2,
    learning_rate=5e-5,

    logging_steps=20,

    save_strategy="no",

    fp16=True
)

### Initializing the Trainer and Starting Model Training

This cell initializes the Hugging Face `Trainer` class and begins the fine-tuning process.

The `Trainer` API provides a high-level interface for training transformer models, handling many tasks automatically such as batching, gradient updates, logging, and GPU utilization.

The trainer is configured with three main components:

- **model** – The GPT-2 Medium model that will be fine-tuned.
- **args** – The previously defined training configuration (`TrainingArguments`) that controls parameters like batch size, learning rate, and number of epochs.
- **train_dataset** – The processed and tokenized dataset used for training.

Once the trainer is initialized, the `trainer.train()` function starts the training process. During training, the model learns to generate appropriate assistant responses based on the instruction-style dataset.

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

> **Note:** The `trainer.train()` line may be commented out to prevent the training process from running automatically when the notebook is executed.  
> Remove the comment (uncomment the line) only when you are ready to start the actual model training.

In [ ]:
# trainer.train()

### Saving the Fine-Tuned Model

This cell saves the final **fine-tuned model** and its corresponding **tokenizer** after training is complete.

The `trainer.save_model()` function stores the trained model weights and configuration in the specified directory. The tokenizer is also saved to the same folder so that the exact vocabulary and special tokens used during training can be preserved.

Saving both the model and tokenizer ensures that the fine-tuned assistant can be easily reloaded later for **inference, further training, or deployment** without needing to repeat the entire training process.

In [ ]:
trainer.save_model("energy_final_model")
tokenizer.save_pretrained("energy_final_model")

### Testing the Fine-Tuned Model

This cell loads the **fine-tuned model and tokenizer** from the saved directory and performs a simple inference test to evaluate the model's behavior after training.

A prompt is constructed using the same conversational format that was used during training, including the special tokens:

- `<|user|>` to represent the user message  
- `<|assistant|>` to indicate where the model should begin generating the response

The prompt is tokenized and converted into tensors that can be processed by the model. The `model.generate()` function is then used to produce a response from the assistant.

Finally, the generated tokens are decoded back into readable text and printed. This step helps verify that the fine-tuned model correctly understands the conversation format and generates an appropriate assistant response.

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model = GPT2LMHeadModel.from_pretrained("energy_final_model")
tokenizer = GPT2Tokenizer.from_pretrained("energy_final_model")

prompt = "<|user|>: what is your name?\n<|assistant|>:"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(**inputs, max_new_tokens=40)

print(tokenizer.decode(output[0]))

In [ ]:
prompt = "<|user|>:Tell me what Energy is\n<|assistant|>:"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(**inputs, max_new_tokens=40)

print(tokenizer.decode(output[0]))